# Sanitized MatrixStoreMinuteFeed Walkthrough

This notebook explains the **runtime guardrail** feed:
`SanitizedMatrixStoreMinuteFeed`.

Use it when you want one extra layer of protection against bad minute bars even after
repairing the store offline.


## What it does

For each symbol, the sanitized feed keeps the **last accepted good price**.

When a new minute arrives:

- if the new price is missing (`NaN`), it is ignored
- if the new price is non-positive, it is rejected
- if the new price moves by more than `max_abs_return` relative to the last accepted
  price, it is rejected
- otherwise it is accepted and becomes the new last-good price

This does **not** rewrite the dataset. It only protects the running backtest from
absurd one-bar moves.


In [ ]:
from pathlib import Path
import pandas as pd
import asyncio

from market_feed.sanitized_matrix_store_1m import SanitizedMatrixStoreMinuteFeed

STORE_DIR = Path('CHANGE_ME_MATRIX_STORE_DIR')
START = '2015-02-01 09:15:00'
END = '2015-02-28 15:30:00'
SYMBOLS = None
MAX_ABS_RETURN = 0.35


## Create the feed instance

`max_abs_return=0.35` means any move larger than **35% in one minute** is rejected.

This is intentionally conservative. It should catch absurd prints without interfering with
normal minute-by-minute market moves.


In [ ]:
feed = SanitizedMatrixStoreMinuteFeed(
    store_dir=str(STORE_DIR),
    start=pd.Timestamp(START).to_pydatetime(),
    end=pd.Timestamp(END).to_pydatetime(),
    symbols=SYMBOLS,
    speed='fast',
    max_abs_return=MAX_ABS_RETURN,
    stats_every_rows=50000,
)
feed


## Peek at a few snapshots

This is just to confirm the feed yields normal sparse snapshots like the original
`MatrixStoreMinuteFeed`, but with suspicious bars filtered out.


In [ ]:
async def collect_first_n(feed, n=5):
    out = []
    async for snap in feed.stream():
        out.append(snap)
        if len(out) >= n:
            break
    return out

snaps = asyncio.run(collect_first_n(feed, n=5))
snaps


## How to use it in the run config

Replace your market-feed config with the sanitized version, or copy these fields into your
existing run module setup.


In [ ]:
sanitized_yaml = """
market_feed:
  type: sanitized_matrix_store_1m
  store_dir: CHANGE_ME_MATRIX_STORE_DIR
  start: "2015-02-01 09:15:00"
  end:   "2015-05-31 15:30:00"
  speed: fast
  max_abs_return: 0.35
  min_price: 1.0e-9
  stats_every_rows: 0
"""

print(sanitized_yaml)


## When to use this feed

Use it when:

- you still see occasional cliffs after offline repairs
- the NAV auditor points to isolated one-bar price shocks
- you want to stop the strategy from trading on obviously broken minute bars

Do **not** treat it as a replacement for the offline repair notebooks. It is a safety
layer, not the primary cleaning pipeline.
